# 17e Quantitative CAM metrics: Base vs HA, last block vs block -4

This notebook runs and analyzes quantitative CAM metrics for the normal 7 class HAM10000 setting.

Goal:

Compare whether choosing an intermediate transformer block improves CAM quality.

Experiments:

1. Base CE, last block
2. Base CE, block -4
3. HA 0.75, last block
4. HA 0.75, block -4

Main question:

Does block -4 give better lesion alignment and/or faithfulness than the final block?

In [ ]:
from pathlib import Path
import subprocess
import shlex
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

QUAL_SEED = 42
NUM_SAMPLES = 100
TOPK_COMPARE = 3

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"

# Use the same CSV you used previously for metric evaluation if available.
# This should contain image_id / image_rel_path / gt_label / mask_rel_path / split.
CANDIDATE_CSVS = [
    HAM_ROOT / "ham_test_for_cam.csv",
    HAM_ROOT / f"ham_test_cam_qualitative_stratified_10_seed{QUAL_SEED}.csv",
]

METRIC_CSV = None
for p in CANDIDATE_CSVS:
    if p.exists():
        METRIC_CSV = p
        break

if METRIC_CSV is None:
    raise FileNotFoundError("No suitable CAM metric CSV found. Please set METRIC_CSV manually.")

CHECKPOINT_ROOT = REPO_ROOT / "external" / "checkpoints3"
BASE_CKPT = CHECKPOINT_ROOT / "checkpoint-best-base.pth"
HA_CKPT = CHECKPOINT_ROOT / "checkpoint-best-HA075.pth"

METRICS_ROOT = REPO_ROOT / "external" / "metrics"
METRICS_ROOT.mkdir(parents=True, exist_ok=True)

IMG_DIR = HAM_ROOT
MASK_ROOT = HAM_ROOT

print("REPO_ROOT:", REPO_ROOT)
print("METRIC_CSV:", METRIC_CSV)
print("BASE_CKPT:", BASE_CKPT)
print("HA_CKPT:", HA_CKPT)
print("METRICS_ROOT:", METRICS_ROOT)

for p in [METRIC_CSV, BASE_CKPT, HA_CKPT]:
    if not p.exists():
        print("[WARN] missing:", p)

In [ ]:
METRIC_JOBS = {
    "Base CE last": {
        "checkpoint": BASE_CKPT,
        "target_block_index": -1,
        "out_dir": METRICS_ROOT / f"metrics_base_last_{NUM_SAMPLES}_topk{TOPK_COMPARE}",
    },
    "Base CE block -4": {
        "checkpoint": BASE_CKPT,
        "target_block_index": -4,
        "out_dir": METRICS_ROOT / f"metrics_base_block4_{NUM_SAMPLES}_topk{TOPK_COMPARE}",
    },
    "HA 0.75 last": {
        "checkpoint": HA_CKPT,
        "target_block_index": -1,
        "out_dir": METRICS_ROOT / f"metrics_ha075_last_{NUM_SAMPLES}_topk{TOPK_COMPARE}",
    },
    "HA 0.75 block -4": {
        "checkpoint": HA_CKPT,
        "target_block_index": -4,
        "out_dir": METRICS_ROOT / f"metrics_ha075_block4_{NUM_SAMPLES}_topk{TOPK_COMPARE}",
    },
}

for name, cfg in METRIC_JOBS.items():
    print(name)
    print("  checkpoint:", cfg["checkpoint"])
    print("  target_block_index:", cfg["target_block_index"])
    print("  out_dir:", cfg["out_dir"])

In [ ]:
def run_command(cmd: list[str], dry_run: bool = False):
    print("\n" + "=" * 100)
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    print("=" * 100)

    if dry_run:
        return

    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


def run_cam_metric_job(
    name: str,
    checkpoint: Path,
    out_dir: Path,
    target_block_index: int,
    dry_run: bool = False,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python", "-m", "scripts.eval_cam_metrics_panderm",
        "--csv", str(METRIC_CSV),
        "--image_col", "image_rel_path",
        "--img_dir", str(IMG_DIR),
        "--gt_col", "gt_label",
        "--checkpoint", str(checkpoint),
        "--checkpoint_model_type", "panderm",
        "--class_preset", "ham",
        "--out_dir", str(out_dir),
        "--num_samples", str(NUM_SAMPLES),
        "--methods", "all",
        "--compare_mode", "gt_topk_non_target",
        "--topk_compare", str(TOPK_COMPARE),
        "--mask_root", str(MASK_ROOT),
        "--mask_col", "mask_rel_path",
        "--target_block_index", str(target_block_index),
    ]

    print(f"\nRunning metric job: {name}")
    run_command(cmd, dry_run=dry_run)

In [ ]:
for name, cfg in METRIC_JOBS.items():
    run_cam_metric_job(
        name=name,
        checkpoint=cfg["checkpoint"],
        out_dir=cfg["out_dir"],
        target_block_index=cfg["target_block_index"],
        dry_run=True,
    )

In [ ]:
for name, cfg in METRIC_JOBS.items():
    run_cam_metric_job(
        name=name,
        checkpoint=cfg["checkpoint"],
        out_dir=cfg["out_dir"],
        target_block_index=cfg["target_block_index"],
        dry_run=False,
    )

In [ ]:
def load_result_set(name, folder):
    folder = Path(folder)

    per_sample_candidates = sorted(folder.glob("per_sample_metrics__*.csv"))
    summary_candidates = sorted(folder.glob("per_method_summary__*.csv"))

    if len(per_sample_candidates) == 0:
        raise FileNotFoundError(f"{name}: no per_sample_metrics__*.csv found in {folder}")
    if len(summary_candidates) == 0:
        raise FileNotFoundError(f"{name}: no per_method_summary__*.csv found in {folder}")

    per_sample_path = max(per_sample_candidates, key=lambda p: p.stat().st_mtime)
    summary_path = max(summary_candidates, key=lambda p: p.stat().st_mtime)

    per_sample = pd.read_csv(per_sample_path)
    summary = pd.read_csv(summary_path)

    per_sample["experiment"] = name
    summary["experiment"] = name
    per_sample["source_file"] = per_sample_path.name
    summary["source_file"] = summary_path.name

    return per_sample, summary


def mean_std_str(mean, std, digits=3):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"